# Stage 00 — Deduplicate → Group → Download → Segment → OCR → Rename

This notebook prepares the patent drawing dataset end-to-end.  
Run cells top-to-bottom; each section's outputs feed the next.

---

## Setup (cells 1–4)

| Cell | What it does | Output |
|------|-------------|--------|
| 1 | Add repo root to `sys.path` | — |
| 2 | Import modules, call `load_config()` | `cfg` dict; prints active paths and `scan_limit` |
| 3 | Build EPO OPS client (skipped in `google_patents` mode) | `epo_client` (or `None`) |
| 4 | Load PatSeer Excel as a metadata dict | `excel_index` — `{patent_id: {title, assignee, …}}` for all ~1,635 rows |

---

## Deduplication (cells 5–6)

The same invention is often published at USPTO, EPO, and WIPO — all appear as separate rows in the Excel.  
`run_deduplication` keeps exactly **one canonical representative per Simple Family ID**.

**Selection priority (lexicographic):**
1. Member with the **most figures / drawings** (maximises image data)
2. Tie-break: **earliest filing date**
3. Tie-break: **office preference** — US > EP > WO > other

**Outputs:**

| File | Contents |
|------|----------|
| `data/deduplicated_patents.csv` | One row per canonical patent (subset of the Excel columns + `review_family` flag) |
| `data/family_map.csv` | One row per family — `family_id`, `canonical_pub_number`, `all_members`, `members_dropped`, `selection_reason` |

`review_family = True` flags families where the figure-count spread among members exceeds 3 — these may represent genuinely different embodiments and deserve a manual look.

---

## Company Grouping + Prototype Inference (cells 7–8)

`run_grouping` adds three semantic columns to the deduplicated DataFrame and builds the processing queue.

**Step A — Company normalisation**  
Each `Assignee` string is mapped to a canonical name via a lookup table of 70+ known variants (Joby, Archer, Lilium, Wisk, Volocopter, EHang, …).  
Variants not in the table are matched with `rapidfuzz` (ratio threshold 85).  
Unmatched assignees → `"Unknown / Independent"`.

**Step B — Prototype generation clustering**  
Within each company group, patent title + abstract are embedded with **PatentSBERTa** (`AI-Growth-Lab/PatentSBERTa`, ~500 MB download on first call).  
**HDBSCAN** (`min_cluster_size = 3`) clusters the embeddings.  
Clusters are sorted by mean filing date and labelled `Prototype_A`, `Prototype_B`, …  
Patents outside any cluster → `"Unclassified"`.

**Columns added:**

| Column | Description |
|--------|-------------|
| `company_canonical` | Normalised company name |
| `prototype_label` | `Prototype_A / _B / …` or `Unclassified` |
| `prototype_cluster_id` | Raw HDBSCAN cluster id (`-1` = noise) |
| `display_order` | Sort key: company → prototype → filing date (ASC) |

**Outputs:**

| Variable / File | Contents |
|-----------------|----------|
| `grouped_df` | Full deduplicated DataFrame + the four columns above |
| `data/grouped_patents.csv` | Same, persisted to disk |
| `patent_ids` | Ordered list of pub-numbers → replaces the raw Excel ordering for the loop below |

A summary table (company × prototype × count × date range) is printed at the end of cell 8.

---

## Per-patent processing loop (cell 9)

Iterates over `patent_ids[:scan_limit]` in `display_order` order.  
Already-processed patents (named figures present) are skipped automatically.

| Sub-step | What it does | Intermediate files | Final output |
|----------|--------------|--------------------|--------------|
| **A — Download** | Fetch drawing pages from Google Patents CDN (or EPO OPS) | `raw/{patent_id}/fig_01.png`, `fig_02.png`, … | — |
| **B — OCR pages** | Run GoFigure alpha-shape OCR on each full page to detect `FIG. X` labels | `page_labels` dict (in memory) | — |
| **C — Description** | Scrape BRIEF DESCRIPTION OF THE DRAWINGS section | — | `text/{patent_id}.txt` |
| **D — Segment** | HR-Net model splits compound pages into individual sub-figure crops | `fig_02_s01.png`, `fig_02_s02.png`, … (originals deleted) | — |
| **E — Rename** | Assign OCR label → permanent name; positional fallback uses description text | — | `raw/{patent_id}/{patent_id}_F001.png`, `_F002A.png`, … |

**Final figure naming convention:**

| Pattern | Meaning |
|---------|---------|
| `{id}_F001.png` | Labeled — OCR found `FIG. 1` |
| `{id}_F002A.png` | Labeled — OCR found `FIG. 2A` |
| `{id}_Fu001.png` | Unlabeled — no `FIG.` label detected (`u` = unlabeled) |
| `{id}_F003_b.png` | Duplicate — two crops both claimed `FIG. 3` (`_b` = duplicate) |

---

## Visualisation (cell 10)

Renders a thumbnail grid (max 6 per row) for every processed patent.  
Figure titles are **green** (labeled) or **red** (unlabeled).

---

## Description text preview (cell 11)

Prints the first 10 lines of each saved `text/{patent_id}.txt` for a quick sanity-check.

---

> **To switch to EPO OPS:** set `extractor.mode: "epo"` in `config.yaml` and fill `EPO_CLIENT_KEY` / `EPO_CLIENT_SECRET` in `.env`.

In [1]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [2]:
from src.config_loader import load_config
from src.extractor import (
    download_drawings,
    get_brief_description,
    load_patseer_excel,
    save_description_text,
)
from src.segmenter import load_segmenter, segment_patent
from src.ocr_labeler import ocr_all_pages, assign_and_rename_crops

cfg  = load_config()
mode = cfg['extractor']['mode']
print(f"Source mode  : {mode}")
print(f"raw_images   : {cfg['paths']['raw_images']}")
print(f"text         : {cfg['paths']['text']}")
print(f"scan_limit   : {cfg['extractor']['scan_limit']}")


Source mode  : google_patents
raw_images   : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Processed Images DataSets & Model Outputs/1635/raw
text         : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Processed Images DataSets & Model Outputs/1635/text
scan_limit   : 10


In [3]:
epo_client = None
if mode == 'epo':
    from src.extractor import build_epo_client
    epo_client = build_epo_client(cfg)
    print('EPO OPS client ready.')
else:
    print(f'Mode = {mode!r} — no EPO credentials needed.')


Mode = 'google_patents' — no EPO credentials needed.


## USPTO Coverage Check (optional)

Queries the [PatentsView API](https://api.patentsview.org/) to determine which US patents
in the PatSeer export are indexed in the USPTO grants database.

| Column | Meaning |
|--------|---------|
| `record_number` | Original PatSeer Record Number (e.g. `US11299268B2`) |
| `stripped_number` | Numeric core sent to PatentsView (e.g. `11299268`) |
| `found_in_uspto` | `True` if the grant number was found (B-series grants only) |
| `patent_title` | Title returned by PatentsView (if found) |
| `patent_date` | Grant date returned by PatentsView (if found) |

**When to run:** set `extractor.run_coverage_check: true` in `config.yaml`.  
Applications (A1/A2) not yet granted will show `found_in_uspto = False`.  
Results are saved to `data/uspto_coverage_check.csv`.

In [4]:
if cfg.get('extractor', {}).get('run_coverage_check', False):
    from scripts.check_uspto_coverage import check_uspto_coverage
    import pandas as pd

    coverage_df = check_uspto_coverage(cfg)

    # ── Display summary table ──────────────────────────────────────────────────
    from IPython.display import display
    print("\nSample rows (first 20):")
    display(
        coverage_df.head(20).style
        .applymap(lambda v: 'background-color: #d4edda' if v is True  else
                            'background-color: #f8d7da' if v is False else '',
                  subset=['found_in_uspto'])
        .hide(axis='index')
    )
    n_found = int(coverage_df['found_in_uspto'].sum())
    print(f"Total US patents: {len(coverage_df)} | Found: {n_found} | Missing: {len(coverage_df) - n_found}")
else:
    print("Coverage check skipped (extractor.run_coverage_check = false in config.yaml).")
    print("Set it to true and re-run to query PatentsView for all US patents.")

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer total  :  1,635
US patents     :    999
Parseable      :    988  (others have unexpected number format)

Querying PatentsView in 40 batch(es) of ≤25…
  [PatentsView] batch query failed: Expecting value: line 1 column 1 (char 0)
  batch   1/40:   0/25 found
  [PatentsView] batch query failed: Expecting value: line 1 column 1 (char 0)
  batch   2/40:   0/25 found
  [PatentsView] batch query failed: Expecting value: line 1 column 1 (char 0)
  batch   3/40:   0/25 found
  [PatentsView] batch query failed: Expecting value: line 1 column 1 (char 0)
  batch   4/40:   0/25 found
  [PatentsView] batch query failed: Expecting value: line 1 column 1 (char 0)
  batch   5/40:   0/25 found
  [PatentsView] batch query failed: Expecting value: line 1 column 1 (char 0)
  batch   6/40:   0/25 found
  [PatentsView] batch query failed: Expecting value: line 1 column 1 (char 0)
  batch   7/40:   0/25 found
  [PatentsView] batch query failed: Expecting value: line 1 column 1 (char 0)
  batch   8/40:

/tmp/ipykernel_45767/3729966688.py:11: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  coverage_df.head(20).style


record_number,stripped_number,found_in_uspto,patent_title,patent_date
US2022267016A1,2022267016,False,None,None
US11787551B1,11787551,False,None,None
US2022234745A1,2022234745,False,None,None
US2015014475A1,2015014475,False,None,None
US2020148347A1,2020148347,False,None,None
US2023312117A1,2023312117,False,None,None
US2022089279A1,2022089279,False,None,None
US2020172235A1,2020172235,False,None,None
US2021403155A1,2021403155,False,None,None
US11485488B1,11485488,False,None,None


Total US patents: 999 | Found: 0 | Missing: 999


In [5]:
excel_index = load_patseer_excel(cfg['paths']['patseer_excel'])
patent_ids  = list(excel_index.keys())
print(f"Indexed {len(patent_ids)} patents from Excel.")


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer Excel: 1635__dataset_26_06_02.xlsx  (1635 rows, 102 columns)
Columns:
  [  0] 'Record Number'
  [  1] 'Title'
  [  2] 'Abstract'
  [  3] 'Description of Drawings'
  [  4] 'CPC'
  [  5] 'PDF Link'
  [  6] 'Record Type'
  [  7] 'Publication/Issue Date'
  [  8] 'Filing/Application Date'
  [  9] 'Estimated Expiry Date'
  [ 10] 'EFAM Earliest Priority Date'
  [ 11] 'EFAM Earliest Publication Date'
  [ 12] 'Priority Details'
  [ 13] 'Priority Dates (All)'
  [ 14] 'Application No.'
  [ 15] 'Priority Country Code'
  [ 16] 'Priority Year'
  [ 17] 'Register Legal Status'
  [ 18] 'Register Legal Status Date'
  [ 19] 'Summary of Invention'
  [ 20] 'Designated States'
  [ 21] 'Active in Designated States'
  [ 22] 'Field of search'
  [ 23] 'Industry'
  [ 24] 'Tech Domain'
  [ 25] 'Tech Sub Domain'
  [ 26] 'Description'
  [ 27] 'Claims'
  [ 28] 'Number Of Claims'
  [ 29] 'No. of Independent Claims'
  [ 30] 'Independent Claims'
  [ 31] 'First Claim'
  [ 32] 'Advantages of Invention'
  [ 33] 'N

## Deduplication + Company / Prototype Grouping

Before queuing patents for download, the 1,635-row PatSeer export is:

1. **Deduplicated** by Simple Family ID — one canonical representative per family selected by:  
   figure count (most) → filing date (earliest) → office (US > EP > WO)
2. **Grouped** by company (fuzzy name normalisation via COMPANY_LOOKUP + rapidfuzz) and  
   prototype generation (PatentSBERTa embeddings + HDBSCAN clustering).

The final processing queue is ordered by `display_order` (company → prototype → date),  
so human review progresses logically through each manufacturer's design lineage.

In [6]:
import pandas as pd
from src.deduplicator import run_deduplication
from src.grouper import run_grouping

# Load raw DataFrame (all columns) separately from the metadata dict above.
# load_patseer_excel() gives a stripped dict; we need the full DataFrame here.
raw_df = pd.read_excel(cfg['paths']['patseer_excel'], dtype=str)
print(f"Raw Excel loaded: {len(raw_df)} rows, {len(raw_df.columns)} columns")

# ── Step 1: Deduplicate by patent family ──────────────────────────────────────
deduplicated_df, family_map = run_deduplication(raw_df, cfg)

# ── Step 2: Group by company + infer prototype generations ────────────────────
# WARNING: first run downloads ~500 MB (PatentSBERTa model from HuggingFace).
grouped_df = run_grouping(deduplicated_df, cfg)

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Raw Excel loaded: 1635 rows, 102 columns
[deduplicator] Columns resolved:
  pub_number  → 'Record Number'
  family_id   → 'Simple Family ID'
  filing_date → 'Filing/Application Date'
  fig_count   → None  (None = column absent, default 0)
  assignee    → 'Assignee'

[deduplicator] Summary
  Input rows          :  1,635
  Unique families     :  1,635
  Patents kept        :  1,635
  Patents dropped     :      0
  Families > 1 member :      0
  Families to review  :      0  (figure count spread > 3)
  Saved: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Processed Images DataSets & Model Outputs/1635/data/deduplicated_patents.csv
  Saved: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Processed Images DataSets & Model Outputs/1635/data/family_map.csv
[grouper] Columns resolved:
  pub_number  → 'Record Number'
  assignee    → 'Assignee'
  filing_date → 'Filing/Application Date'
  title       → 'Title'
  abstract    → 'Abstract'

[grouper] Normalising company names …
  Known companies   

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Batches:   0%|          | 0/52 [00:00<?, ?it/s]


[grouper] Clustering by company group …
  AMSL Innovations                            4 patents → 0 prototype(s), 4 unclassified
  Aerhart                                     3 patents → 0 prototype(s), 3 unclassified
  AeroVironment                               3 patents → 0 prototype(s), 3 unclassified
  Aeronext                                    9 patents → 2 prototype(s), 1 unclassified
  Airbus                                     26 patents → 3 prototype(s), 13 unclassified
  Alphabet / X                                3 patents → 0 prototype(s), 3 unclassified
  Amazon                                      8 patents → 0 prototype(s), 8 unclassified
  Anduril                                     3 patents → 0 prototype(s), 3 unclassified
  Archer Aviation                            11 patents → 0 prototype(s), 11 unclassified
  Ascendance Flight Technologies              9 patents → 0 prototype(s), 9 unclassified
  Aurora Flight Sciences                     14 patents → 2 prototy

In [7]:
# ── Override patent_ids with the deduplicated, grouped ordering ───────────────
# grouped_df preserves the original pub-number column name from the Excel.
_PUB_VARIANTS = [
    'Publication Number', 'Pub. No.', 'Patent Number', 'Record Number',
]
_pub_col = next((c for c in _PUB_VARIANTS if c in grouped_df.columns), None)
if _pub_col is None:
    raise KeyError(
        f"Cannot find a publication-number column in grouped_df. "
        f"Expected one of {_PUB_VARIANTS}. Got: {list(grouped_df.columns[:10])}"
    )

_DATE_VARIANTS = [
    'Filing Date', 'Application Date', 'App. Date', 'Filing/Application Date',
]
_date_col = next((c for c in _DATE_VARIANTS if c in grouped_df.columns), None)

patent_ids = grouped_df.sort_values('display_order')[_pub_col].tolist()
print(f"Processing queue : {len(patent_ids)} patents (deduplicated, ordered by company/prototype/date)")
print(f"  (was {len(raw_df)} rows before deduplication)")
print()

# ── Print company | prototype | count | date-range summary ───────────────────
_agg: dict = {'count': (_pub_col, 'count')}
if _date_col:
    _agg['earliest'] = (_date_col, lambda x: str(x.dropna().min())[:7] if x.notna().any() else '')
    _agg['latest']   = (_date_col, lambda x: str(x.dropna().max())[:7] if x.notna().any() else '')

_summary = (
    grouped_df
    .groupby(['company_canonical', 'prototype_label'])
    .agg(**_agg)
    .reset_index()
)

print(f"{'Company':<38} {'Prototype':<16} {'#':>5}  Date range")
print("─" * 78)
for _, row in _summary.iterrows():
    date_range = ''
    if _date_col:
        date_range = f"{row.get('earliest', '')} – {row.get('latest', '')}"
    print(
        f"{str(row['company_canonical']):<38} "
        f"{str(row['prototype_label']):<16} "
        f"{int(row['count']):>5}  {date_range}"
    )

Processing queue : 1635 patents (deduplicated, ordered by company/prototype/date)
  (was 1635 rows before deduplication)

Company                                Prototype            #  Date range
──────────────────────────────────────────────────────────────────────────────
AMSL Innovations                       Unclassified         4  2018-09 – 2025-04
Aerhart                                Unclassified         3  2021-07 – 2022-08
AeroVironment                          Unclassified         3  2013-09 – 2017-06
Aeronext                               Prototype_A          5  2018-01 – 2021-03
Aeronext                               Prototype_B          3  2018-09 – 2020-09
Aeronext                               Unclassified         1  2015-05 – 2015-05
Airbus                                 Prototype_A          3  2014-08 – 2016-05
Airbus                                 Prototype_B          6  2018-02 – 2019-12
Airbus                                 Prototype_C          4  2012-09 – 2022

In [8]:
seg_model = load_segmenter(cfg)
print('HR-Net segmentation model ready.')


I0000 00:00:1780748003.783068   45767 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
W0000 00:00:1780748005.047186   45767 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


HR-Net loaded from model-hrnet-new1.h5
HR-Net segmentation model ready.


In [ ]:
scan_limit = cfg['extractor']['scan_limit']
raw_dir    = cfg['paths']['raw_images']
text_dir   = cfg['paths']['text']

results = {}
errors  = []

for patent_id in patent_ids[:scan_limit]:
    print(f"\n{'─'*60}")
    print(f"  {patent_id}")
    print(f"{'─'*60}")

    patent_dir = raw_dir / patent_id

    # Skip if already fully processed
    existing_final = []
    if patent_dir.exists():
        existing_final = sorted(patent_dir.glob(f"{patent_id}_F*.png"))
        existing_final += sorted(patent_dir.glob(f"{patent_id}_Fu*.png"))
    if existing_final:
        print(f"  Already processed -- {len(existing_final)} named figures")
        results[patent_id] = existing_final
        continue

    # A: Download drawing pages
    pages = download_drawings(patent_id, cfg, raw_dir, epo_client)
    if not pages:
        errors.append((patent_id, 'no drawings downloaded'))
        results[patent_id] = []
        continue

    # B: OCR original pages BEFORE segmentation
    print(f"  OCR original pages ({len(pages)})...")
    page_labels = ocr_all_pages(pages, cfg)

    # C: Fetch BRIEF DESCRIPTION (needed before renaming for positional fallback)
    desc = get_brief_description(patent_id, cfg, epo_client)
    if desc:
        txt_path = save_description_text(patent_id, desc, text_dir)
        print(f"  Description: {len(desc.splitlines())} lines -> {txt_path.name}")
    else:
        print(f"  Description: not found")

    # D: Segment pages into sub-figures
    page_to_crops = segment_patent(patent_id, cfg, raw_dir, seg_model)

    # E: Assign labels (crop OCR + page fallback + description positional) and rename
    named_figures = assign_and_rename_crops(
        patent_id, page_to_crops, page_labels, cfg, description_text=desc or ''
    )
    results[patent_id] = named_figures

total = sum(len(v) for v in results.values())
print(f"\n{'='*60}")
print(f"  Patents processed : {len(results)}")
print(f"  Total named figs  : {total}")
if errors:
    print(f"  Issues ({len(errors)}):")
    for pid, reason in errors:
        print(f"    {pid} -> {reason}")



────────────────────────────────────────────────────────────
  US2020223542A1
────────────────────────────────────────────────────────────
  Drawing pages: 12
      ✓ fig_01.png  (84 kB)
      ✓ fig_02.png  (148 kB)
      ✓ fig_03.png  (141 kB)
      ✓ fig_04.png  (127 kB)
      ✓ fig_05.png  (48 kB)
      ✓ fig_06.png  (47 kB)
      ✓ fig_07.png  (38 kB)
      ✓ fig_08.png  (40 kB)
      ✓ fig_09.png  (38 kB)
      ✓ fig_10.png  (229 kB)
      ✓ fig_11.png  (88 kB)
      ✓ fig_12.png  (35 kB)
  OCR original pages (12)...
  Page-level OCR: 0/12 pages with FIG. labels (0 labels total)
  Description: 21 lines -> US2020223542A1.txt
    fig_02.png → 2 sub-figures
    fig_03.png → 2 sub-figures
    fig_04.png → 2 sub-figures
    fig_05.png → 2 sub-figures
    fig_06.png → 3 sub-figures
    fig_07.png → 2 sub-figures
    fig_08.png → 2 sub-figures
    fig_09.png → 8 sub-figures
    fig_10.png → 2 sub-figures
    fig_11.png → 2 sub-figures
  Segmentation: 12 pages → 29 figures (10 page(s) sp

## Retroactive relabeling — fix fully-unlabeled patents

After the main loop, some patent folders may contain **only `_Fu*.png` files** — meaning OCR
found no `FIG. X` labels anywhere and the positional fallback was also skipped
(because HR-Net split at least one page, making the intra-page order uncertain).

`relabel_all_unlabeled` scans every patent folder and, for those that are completely
unlabeled, attempts a **positional match** using the saved description `.txt`:

1. Collect `_Fu*.png` files (sorted by filename)
2. Load `text/{patent_id}.txt`; if absent, try to fetch the description from the web
3. Parse the *Brief Description of the Drawings* section → ordered FIG. keys
4. Rename positionally: `_Fu001.png → _F001.png`, `_Fu002.png → _F002A.png`, etc.

**Guard:** only fires when **all** files are `_Fu*`; partially-labeled patents are left
untouched so as not to disturb correctly-named figures.

In [ ]:
from src.ocr_labeler import relabel_all_unlabeled

relabel_all_unlabeled(
    cfg,
    raw_dir=cfg['paths']['raw_images'],
    text_dir=cfg['paths']['text'],
    epo_client=epo_client,
)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

THUMB   = (200, 200)
MAX_COL = 6

for patent_id, figure_paths in results.items():
    if not figure_paths:
        continue
    labeled   = [p for p in figure_paths if '_F' in p.name and '_Fu' not in p.name]
    unlabeled = [p for p in figure_paths if '_Fu' in p.name]
    print(f"\n{'─'*60}")
    print(f"  {patent_id}  --  {len(labeled)} labeled  |  {len(unlabeled)} unlabeled")
    print(f"{'─'*60}")
    for row_start in range(0, len(figure_paths), MAX_COL):
        chunk = figure_paths[row_start:row_start + MAX_COL]
        fig, axes = plt.subplots(1, len(chunk), figsize=(3 * len(chunk), 3))
        if len(chunk) == 1:
            axes = [axes]
        for ax, p in zip(axes, chunk):
            try:
                img = Image.open(p)
                img.thumbnail(THUMB)
                ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
            except Exception:
                ax.text(0.5, 0.5, 'missing', ha='center', va='center')
            color = 'red' if '_Fu' in p.name else 'green'
            ax.set_title(p.stem.split('_', 1)[-1], fontsize=7, color=color)
            ax.axis('off')
        plt.tight_layout()
        plt.show()


In [ ]:
for patent_id in results:
    txt_path = text_dir / f"{patent_id}.txt"
    if not txt_path.exists():
        print(f"\n{patent_id}: no description text saved")
        continue
    lines = txt_path.read_text(encoding='utf-8').splitlines()
    print(f"\n{'─'*60}")
    print(f"  {patent_id}  ({len(lines)} lines)")
    print(f"{'─'*60}")
    for line in lines[:10]:
        print(f"  {line}")
    if len(lines) > 10:
        print(f"  ... ({len(lines) - 10} more lines)")


In [ ]:
"""
Integrity check for the deduplication + grouping outputs.
Run this cell AFTER cells 5–8 have produced the three CSVs.
"""
import pandas as pd

data_dir  = cfg['paths']['data']
dedup     = pd.read_csv(data_dir / 'deduplicated_patents.csv', dtype=str)
fmap      = pd.read_csv(data_dir / 'family_map.csv',           dtype=str)
grp       = pd.read_csv(data_dir / 'grouped_patents.csv',      dtype=str)
raw_df    = pd.read_excel(cfg['paths']['patseer_excel'],        dtype=str)

PUB_COL    = 'Record Number'       # pub-number column in PatSeer export
FAMILY_COL = 'Simple Family ID'   # family-ID column in PatSeer export

# ── 1. Basic counts ───────────────────────────────────────────────────────────
print("=" * 60)
print("1. Basic counts")
print("=" * 60)
print(f"  Original Excel rows        : {len(raw_df):>6,}")
print(f"  Deduplicated rows          : {len(dedup):>6,}  (patents kept)")
print(f"  Dropped                    : {len(raw_df) - len(dedup):>6,}")
print(f"  Unique families (fmap)     : {len(fmap):>6,}")
print(f"  Patents in grouped CSV     : {len(grp):>6,}")
print()

# ── 2. No duplicate canonical pub-numbers in dedup CSV ───────────────────────
n_dup_pub = dedup.duplicated(subset=[PUB_COL]).sum()
print("=" * 60)
print("2. Dedup CSV — duplicate canonical pub-numbers")
print("=" * 60)
print(f"  Duplicates (should be 0)   : {n_dup_pub}")
if n_dup_pub:
    print("  !! PROBLEM — these pub-numbers appear more than once:")
    print(dedup[dedup.duplicated(subset=[PUB_COL], keep=False)][[PUB_COL, FAMILY_COL]])
print()

# ── 3. review_family flag distribution ───────────────────────────────────────
n_review = (dedup['review_family'].str.lower() == 'true').sum()
print("=" * 60)
print("3. review_family flag (figure-count spread > 3)")
print("=" * 60)
print(f"  Flagged families           : {n_review}")
if n_review:
    review_rows = dedup[dedup['review_family'].str.lower() == 'true']
    print(review_rows[[PUB_COL, FAMILY_COL, 'review_family']].head(10).to_string(index=False))
print()

# ── 4. Conservation check — all original pub-numbers accounted for ────────────
print("=" * 60)
print("4. Conservation check")
print("=" * 60)
original_pubs = set(raw_df[PUB_COL].astype(str).str.strip())

accounted_canonical = set(fmap['canonical_pub_number'].astype(str).str.strip())

accounted_dropped = set()
for cell in fmap['members_dropped'].dropna():
    for pub in str(cell).split('; '):
        pub = pub.strip()
        if pub:
            accounted_dropped.add(pub)

accounted_all = accounted_canonical | accounted_dropped
lost          = original_pubs - accounted_all
extra         = accounted_all - original_pubs

print(f"  Original pub-numbers       : {len(original_pubs):>6,}")
print(f"  Accounted for (canon+drop) : {len(accounted_all):>6,}")
print(f"  LOST        (should be 0)  : {len(lost):>6,}")
print(f"  EXTRA       (should be 0)  : {len(extra):>6,}")
if lost:
    print(f"  Lost sample: {list(lost)[:5]}")
if extra:
    print(f"  Extra sample: {list(extra)[:5]}")
print()

# ── 5. Selection reason breakdown ─────────────────────────────────────────────
print("=" * 60)
print("5. Selection reason breakdown (family_map)")
print("=" * 60)
# Normalise to the reason prefix
fmap['_reason_key'] = fmap['selection_reason'].str.split('=').str[0]
print(fmap['_reason_key'].value_counts().to_string())
print()

# ── 6. Family size distribution ───────────────────────────────────────────────
print("=" * 60)
print("6. Family size distribution")
print("=" * 60)
fmap['family_size'] = pd.to_numeric(fmap['family_size'], errors='coerce')
size_dist = fmap['family_size'].value_counts().sort_index()
for sz, cnt in size_dist.items():
    bar = '█' * min(int(cnt / max(size_dist) * 30), 30)
    print(f"  {int(sz):>3} member(s) : {cnt:>5,}  {bar}")
print()

# ── 7. Company / prototype summary ────────────────────────────────────────────
print("=" * 60)
print("7. Company / prototype summary (grouped_patents.csv)")
print("=" * 60)
grp_summary = (
    grp.groupby(['company_canonical', 'prototype_label'])
       .size()
       .reset_index(name='count')
       .sort_values(['company_canonical', 'prototype_label'])
)
print(grp_summary.to_string(index=False))
